# Phase 3: Multi-Channel Candidate Blocker & Provenance Engine
## Amazon ML Challenge 2026 — Multilingual Business Entity Resolution

This notebook executes the full Phase 3 pipeline:
1. **Step 3.1 (Country-Aware Token IDF):** Computes mathematically rigorous $IDF(t)$ for name and address tokens per country.
2. **Step 3.2 (8 Independent Retrieval Channels):** Queries `name_core`, `name_tokens`, `name_containment`, `acronym_bidir`, `addr_rare_tokens`, `addr_numeric`, `postal_match`, and `phonetic_anchor`.
3. **Step 3.3 (Candidate Set UNION & Provenance Tracking):** Merges candidates across channels without early drop and attaches 8-bit hit mask.
4. **Step 3.4 (Diagnostic Recall@K & Oracle F0.5 Curve):** Sweeps $K \in [5..100]$, measures Oracle F0.5 ceilings, and generates candidate pairs for Phase 4.

In [ ]:
# Cell 1: Environment & Module Setup
import os
import sys
import time
import json
from pathlib import Path
import polars as pl

# Locate project root
current_path = Path.cwd().resolve()
while current_path.parent != current_path:
    if (current_path / "dataset").exists() or (current_path / "code").exists():
        project_root = current_path
        break
    current_path = current_path.parent
else:
    project_root = Path.cwd().resolve()

ber_code_dir = project_root / "code" / "business_entity_resolution"
if str(ber_code_dir) not in sys.path:
    sys.path.insert(0, str(ber_code_dir))

print(f"Project Root: {project_root}")
print(f"BER Code Dir: {ber_code_dir}")

from src.config import PROCESSED_DIR, DATASET_DIR, TRAIN_DIR
from src.country_idf import CountryIDFComputer
from src.blocking_channels import CountryMultiChannelIndex
from src.blocking import MultiChannelBlocker
from src.ablation_blocking import evaluate_blocking_benchmark

print("All Phase 3 modules imported successfully!")

In [ ]:
# Cell 2: Step 3.4 — Run Candidate Blocker Benchmark & Oracle F0.5 Evaluation Curve
benchmark_results = evaluate_blocking_benchmark(
    sample_size=30000,
    k_list=[5, 10, 15, 20, 25, 35, 50, 75, 100]
)

In [ ]:
# Cell 3: Step 3.2 — Generate Fold 0 Validation Candidate Pairs with Provenance (K = 50)
from src.run_phase3 import run_phase3_pipeline
run_phase3_pipeline(max_k=50, run_benchmark=False, generate_val_pairs=True)

In [ ]:
# Cell 4: Inspect Generated Candidate Pairs & Provenance Schema
cand_df = pl.read_parquet(PROCESSED_DIR / "val_candidate_pairs.parquet").head(10)
print("Sample Candidate Pairs with Provenance:")
print(cand_df)